# 03 — Chunking and Document Loading

## Why this notebook exists

Notebook 02 ranked whole documents by embedding each one as a single vector. That worked because our documents are tiny and each is about one thing. Real documents aren't: a support page covers warranty *and* hours *and* spare-part depots, and a PDF manual runs for pages. Squash all of that into one vector and it blurs — a question about spare parts and a question about warranty both match the same averaged document, and specific details get washed out.

The fix is to break documents into smaller, focused **chunks** and embed those, so retrieval can return the exact passage that answers a question. But first we have to *get the documents in* — including a PDF, which needs real extraction. This notebook does both: load the corpus (markdown + a PDF, via `pypdf`) into text with `source`/`page` metadata, split it into well-sized overlapping chunks, and show chunk-level retrieval returning the precise passage — with a citation — instead of a whole averaged document.

Requires an `OPENAI_API_KEY` (we embed chunks at the end to show the payoff). Still no vector database — just NumPy.

## What you'll learn

- How to **load** a mixed corpus — markdown files and a **PDF** (with `pypdf`) — into a uniform list of text units carrying `source` and `page` metadata.
- Why a single embedding per document is too coarse, and why **chunking** improves retrieval granularity.
- Three **chunking strategies**: naive fixed-size character splits, **token-based** splits (`tiktoken`), and **recursive/structure-aware** splits (`langchain-text-splitters`) — and the role of **chunk overlap**.
- How to attach **metadata** (`source`, `page`, `chunk_id`) to every chunk so retrieved passages can be cited.
- How **chunk-level retrieval** returns the exact relevant passage — the fix for notebook 02's whole-document blur.

## 1. Setup

We reuse notebook 02's embedding toolkit — `embed`, `embed_many`, and `cosine` — re-declared inline so this notebook stands alone. We load `OPENAI_API_KEY` from the environment (and from a local `.env` if `python-dotenv` is installed). Document loading and chunking themselves need no API; the key is used only at the end (Section 6) to embed chunks and show the retrieval payoff.

In [ ]:
import os

# Optional: load a local .env if python-dotenv is installed. Real env vars win.
try:
    from dotenv import load_dotenv
    load_dotenv()
except ImportError:
    pass

# ── Key guard ──────────────────────────────────────────────────────────────
if not os.environ.get("OPENAI_API_KEY"):
    print("=" * 60)
    print("OPENAI_API_KEY is not set.")
    print("=" * 60)
    print()
    print("This notebook embeds chunks at the end (Section 6) to show the")
    print("retrieval payoff. Loading and chunking themselves need no key.")
    print()
    print("Set it and restart the kernel:")
    print("  export OPENAI_API_KEY=sk-...")
    raise SystemExit("Set OPENAI_API_KEY and restart the kernel to continue.")

print("OPENAI_API_KEY set ✓")

In [ ]:
import numpy as np
from openai import OpenAI

EMBED_MODEL = "text-embedding-3-small"

openai_client = OpenAI()  # reads OPENAI_API_KEY from the environment


def embed(text: str) -> np.ndarray:
    """Embed a single string into a NumPy vector."""
    resp = openai_client.embeddings.create(model=EMBED_MODEL, input=text)
    return np.array(resp.data[0].embedding, dtype=np.float32)


def embed_many(texts: list) -> np.ndarray:
    """Embed a list of strings in ONE API call; returns a (len(texts), dim) matrix."""
    resp = openai_client.embeddings.create(model=EMBED_MODEL, input=texts)
    return np.array([d.embedding for d in resp.data], dtype=np.float32)


def cosine(a: np.ndarray, b: np.ndarray) -> float:
    """Cosine similarity between two vectors (higher = more similar)."""
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))


print("Setup OK")
print(f"Embedding model: {EMBED_MODEL}")

## 2. Loading Documents

Retrieval needs raw text, but our corpus is mixed: markdown files and a PDF. We load both into one uniform shape — a list of dicts, each with `source` (filename), `page` (a 1-based page number for PDFs, or `None` for markdown), and `text`.

Markdown is just a file read. The PDF needs real extraction: `pypdf` opens it and gives us the text of each page separately, which is why PDF pages become separate units — page numbers are useful metadata for citing where an answer came from.

In [ ]:
from pathlib import Path
from pypdf import PdfReader

# Resolve the data folder whether the kernel runs from rag/ or from the repo root.
DATA_DIR = Path("data")
if not DATA_DIR.exists():
    DATA_DIR = Path("rag/data")


def load_documents(data_dir: Path) -> list:
    """Load every .md and .pdf in data_dir into a uniform list of text units.

    Each unit is a dict: {"source": filename, "page": int|None, "text": str}.
    Markdown files become one unit (page=None); each PDF page becomes its own
    unit (page=1, 2, ...), so we can cite the page an answer came from.
    """
    docs = []
    for path in sorted(data_dir.glob("*.md")):
        docs.append({"source": path.name, "page": None, "text": path.read_text(encoding="utf-8")})
    for path in sorted(data_dir.glob("*.pdf")):
        reader = PdfReader(str(path))
        for i, page in enumerate(reader.pages):
            docs.append({"source": path.name, "page": i + 1, "text": page.extract_text() or ""})
    return docs


documents = load_documents(DATA_DIR)

# Fail clearly if nothing loaded, instead of a confusing error later.
if not documents:
    raise FileNotFoundError(
        f"No documents found in {DATA_DIR}/. Run notebook 01 (and notebook 03's "
        "PDF-authoring step) first."
    )

print(f"Loaded {len(documents)} document units (markdown files + PDF pages):\n")
for d in documents:
    loc = d["source"] if d["page"] is None else f"{d['source']} p.{d['page']}"
    print(f"  {loc:<34} {len(d['text']):>5} chars")

In [ ]:
# The PDF is where loading does real work — show what pypdf extracted.
pdf_units = [d for d in documents if d["source"].endswith(".pdf")]
print(f"The PDF contributed {len(pdf_units)} page-units.\n")
print("Page 1 extracted text:")
print("-" * 60)
print(pdf_units[0]["text"])

## 3. Why Chunk?

Now that documents are loaded, why not just embed each one whole, like notebook 02 did? Two reasons:

1. **Granularity.** `support.md` covers warranty, support hours, *and* spare-part depots. As one vector it's an average of all three topics, so it matches every related question only weakly and never strongly. Split it into a warranty chunk, an hours chunk, and a depots chunk, and a question about spare parts matches the depots chunk *precisely*.
2. **Length.** Embedding models have an input limit and lose fidelity on long text. A 30-page PDF cannot be one useful vector. Chunks keep each embedded unit short and focused.

So we split documents into **chunks**: small spans of text, each embedded on its own. The art is choosing *where* to split. Too large and you're back to the blurring problem; too small and a chunk loses the context needed to make sense. The next section compares strategies.

> **Gotcha:** A good chunk is semantically self-contained — it should still make sense on its own when a model reads it in isolation. Splitting blindly every N characters often cuts a sentence (or a number and its unit) in half, which is why structure-aware splitting and overlap matter.

## 4. Chunking Strategies

We'll compare three ways to split text, using `support.md` as the sample:

1. **Naive fixed-size** — slice every N characters. Simple, but it cuts blindly through words and sentences.
2. **Recursive / structure-aware** — `RecursiveCharacterTextSplitter` from `langchain-text-splitters` tries to split on paragraph, then sentence, then word boundaries, so chunks end at natural breaks.
3. **Token-based** — split by *token* count (what the model and the embedding API actually measure) rather than characters, using a `tiktoken` encoder.

We also show **overlap**: letting consecutive chunks share a little text so a fact that straddles a boundary isn't lost.

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

sample = next(d for d in documents if d["source"] == "support.md")["text"]
print(f"Sample: support.md ({len(sample)} chars)\n")

# 1. Naive fixed-size character chunks
def fixed_chunks(text: str, size: int = 200) -> list:
    return [text[i:i + size] for i in range(0, len(text), size)]

naive = fixed_chunks(sample, 200)
print(f"Naive fixed 200-char chunks: {len(naive)}")
print(f"  chunk 0 ends: ...{naive[0][-45:]!r}")
print("  ^ note how it can cut mid-word / mid-sentence\n")

# 2. Recursive / structure-aware chunks (respects paragraph & sentence breaks)
splitter = RecursiveCharacterTextSplitter(chunk_size=200, chunk_overlap=40)
recursive = splitter.split_text(sample)
print(f"Recursive 200-char chunks (overlap 40): {len(recursive)}")
print(f"  chunk 0 ends: ...{recursive[0][-45:]!r}")
print("  ^ ends on a natural boundary")

In [ ]:
import tiktoken

enc = tiktoken.encoding_for_model("gpt-4o-mini")

# 3. Token-based splitting: chunk by token count, not character count.
tok_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    chunk_size=60, chunk_overlap=15
)
tok_chunks = tok_splitter.split_text(sample)
print(f"Token-based chunks (~60 tokens, overlap 15): {len(tok_chunks)}")
for i, c in enumerate(tok_chunks):
    print(f"  chunk {i}: {len(enc.encode(c)):>3} tokens, {len(c):>3} chars")

# Overlap in action: trailing text of one chunk reappears at the start of the next.
if len(recursive) > 1:
    print("\nOverlap carries context across the boundary:")
    print(f"  end of recursive chunk 0  : ...{recursive[0][-40:]!r}")
    print(f"  start of recursive chunk 1: {recursive[1][:40]!r}...")